# Demo: NRP from a terminal

1. **The cluster**, from kubectl
2. **The LLM gateway**, with curl
3. **A GPU job**
4. **Your own agent**, in Python: [agent.ipynb](agent.ipynb), while the job runs

This notebook runs on the **Bash** kernel on the training JupyterHub, where `kubectl` already points at the `nrp-training-k8s` namespace and `OPENAI_API_KEY` holds an LLM token.

> 📘 [Getting started](https://nrp.ai/documentation/userdocs/start/getting-started/) · [LLM API access](https://nrp.ai/documentation/userdocs/ai/llm-managed/api-access/) · [GPU pods](https://nrp.ai/documentation/userdocs/running/gpu-pods/)

## 1. The cluster

Topology and hardware are labels on every node. `-L` shows a label as a column, so this counts the nodes in each region (the region is column 6):

In [ ]:
kubectl get nodes -L topology.kubernetes.io/region --no-headers | awk '{print $6}' | sort | uniq -c | sort -rn

The same for GPU nodes and their GPU model. `-l` keeps only the nodes that have the label:

In [ ]:
kubectl get nodes -l nvidia.com/gpu.product -L nvidia.com/gpu.product --no-headers | awk '{print $6}' | sort | uniq -c | sort -rn | head

## 2. The LLM gateway

The model names the gateway serves:

In [ ]:
curl -s https://ellm.nrp-nautilus.io/v1/models -H "Authorization: Bearer $OPENAI_API_KEY" | jq -r '.data[].id'

Ask `qwen3` a question. `jq` picks out which model answered, the token counts and the answer:

In [ ]:
curl -s https://ellm.nrp-nautilus.io/v1/chat/completions \
  -H "Authorization: Bearer $OPENAI_API_KEY" -H "Content-Type: application/json" \
  -d '{"model": "qwen3", "reasoning_effort": "low",
       "messages": [{"role": "user", "content": "In two sentences: what does a network operations center do?"}]}' \
  | jq '{model, usage, answer: .choices[0].message.content}'

The rate limit is in the response headers (`-i` prints them):

In [ ]:
curl -si https://ellm.nrp-nautilus.io/v1/chat/completions \
  -H "Authorization: Bearer $OPENAI_API_KEY" -H "Content-Type: application/json" \
  -d '{"model": "qwen3", "reasoning_effort": "low", "messages": [{"role": "user", "content": "Say hi."}]}' \
  | grep -i ratelimit

Now ask for a **task** instead of a model. `coding` is a list of models on the gateway, tried in order:

In [ ]:
curl -s https://ellm.nrp-nautilus.io/v1/chat/completions \
  -H "Authorization: Bearer $OPENAI_API_KEY" -H "Content-Type: application/json" \
  -d '{"model": "coding", "reasoning_effort": "low",
       "messages": [{"role": "user", "content": "Write one kubectl command that counts nodes per value of the label topology.kubernetes.io/region. Code only."}]}' \
  | jq -r '.model, .choices[0].message.content'

Compare its answer with the region count in part 1, and read it before you run it.

## 3. A GPU job

`../yamls/gpu-job.yaml` asks for one GPU with more than 20 GB of memory, Ampere or newer, trains a small network on handwritten digits (MNIST), and exits. The top of the file:

In [ ]:
head -40 ../yamls/gpu-job.yaml

Everyone's job needs its own name. **Edit this cell**: put your short name in place of `changeme` (lowercase letters and digits), then run it.

In [ ]:
ME=changeme

Submit the job, with your name in place of `<username>`:

In [ ]:
sed "s/<username>/$ME/" ../yamls/gpu-job.yaml | kubectl apply -f -

Check on it. **Pending** means it's waiting for a free GPU; run the cell again until it says **Running**.

In [ ]:
kubectl get pods -l job-name=gpu-train-$ME -o wide

While it starts, build an agent in **[agent.ipynb](agent.ipynb)** (part 4), then come back here.

Once it's Running, stream its output:

In [ ]:
kubectl logs -f job/gpu-train-$ME

Still Pending? The events at the end of `describe` say why, usually that no matching GPU is free:

In [ ]:
kubectl describe pod -l job-name=gpu-train-$ME | tail -15

Clean up. (The job also deletes itself an hour after it finishes.)

In [ ]:
kubectl delete job gpu-train-$ME

## Next

- Your own namespace: [get-started.nrp-nautilus.io](https://get-started.nrp-nautilus.io/), then [kubectl and the NRP kubeconfig](https://nrp.ai/documentation/userdocs/start/getting-started/).
- An LLM token: [nrp.ai/llmtoken](https://nrp.ai/llmtoken). The [models page](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) lists what each model is good for.
- Build images in CI: [Building in GitLab](https://nrp.ai/documentation/userdocs/development/gitlab/).
- How it's run: the [admin guide](https://nrp.ai/documentation/admindocs/).